# Session 2: measuring a classifier
**A. Real-world problem.** A screening test says "positive". How likely is it that the
person really has the condition? A test that is 99% accurate can still be wrong most of the
time it says positive, if the condition is rare. Today you write two classifiers (a decision
tree and nearest neighbours), build the confusion matrix, and compute this effect with
Bayes' rule. (General education only, with made-up numbers: never personal medical advice.)

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: synthetic data with a curved boundary
Two interleaved half-moons plus noise. A straight line cannot separate them well, so we need
classifiers with bendy boundaries.

In [ ]:
def make_moons(n_each=100, noise=0.3, seed=0):
    rng = np.random.default_rng(seed)
    t = rng.uniform(0, np.pi, n_each)
    a = np.stack([np.cos(t), np.sin(t)], axis=1)
    t2 = rng.uniform(0, np.pi, n_each)
    b = np.stack([1 - np.cos(t2), 0.5 - np.sin(t2)], axis=1)
    X = np.vstack([a, b]) + rng.normal(0, noise, (2 * n_each, 2))
    y = np.r_[np.zeros(n_each, int), np.ones(n_each, int)]
    return X, y

## Step 2: Gini impurity, the "how mixed is this group" number
For a group where a fraction p1 is class 1 and p0 = 1 - p1 is class 0:

    Gini = 1 - p0^2 - p1^2

A pure group has Gini 0. A 50/50 group has Gini 1 - 0.25 - 0.25 = 0.5, the worst case for
two classes. A split is good when the weighted Gini of the two halves is small.

In [ ]:
def gini(y):
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    p1 = float(np.mean(y == 1))
    return 1.0 - p1 * p1 - (1.0 - p1) ** 2


def best_split(X, y):
    """Search every feature and every midpoint; return (feature, threshold, weighted_gini).
    Returns (None, None, gini(y)) when no split separates anything."""
    X, y = np.asarray(X, float), np.asarray(y)
    n = len(y)
    best = (None, None, gini(y))
    for j in range(X.shape[1]):
        order = np.argsort(X[:, j], kind="stable")
        xs, ys = X[order, j], y[order]
        ones_left = np.cumsum(ys == 1)
        total_ones = ones_left[-1]
        for i in range(1, n):
            if xs[i] == xs[i - 1]:
                continue
            nl, nr = i, n - i
            p_l = ones_left[i - 1] / nl
            p_r = (total_ones - ones_left[i - 1]) / nr
            g = (nl * (1 - p_l ** 2 - (1 - p_l) ** 2) + nr * (1 - p_r ** 2 - (1 - p_r) ** 2)) / n
            if g < best[2] - 1e-12:
                best = (j, 0.5 * (xs[i] + xs[i - 1]), g)
    return best

## Step 3: a tree is repeated splitting
Split the data at the best question, then split each half again, down to `max_depth`.
A tree of depth 1 is called a *stump*. A leaf predicts the fraction of class 1 inside it
(a score between 0 and 1); we call it class 1 when that is above 0.5.

In [ ]:
def build_tree(X, y, max_depth):
    X, y = np.asarray(X, float), np.asarray(y)
    if max_depth == 0 or gini(y) == 0.0:
        return {"leaf": True, "p1": float(np.mean(y == 1)), "n": int(len(y))}
    j, t, g = best_split(X, y)
    if j is None:
        return {"leaf": True, "p1": float(np.mean(y == 1)), "n": int(len(y))}
    m = X[:, j] <= t
    return {"leaf": False, "feature": j, "threshold": float(t),
            "left": build_tree(X[m], y[m], max_depth - 1), "right": build_tree(X[~m], y[~m], max_depth - 1)}


def tree_score(node, X):
    """Fraction of class 1 in the leaf each row falls into."""
    X = np.asarray(X, float)
    out = np.empty(len(X))
    for i, row in enumerate(X):
        nd = node
        while not nd["leaf"]:
            nd = nd["left"] if row[nd["feature"]] <= nd["threshold"] else nd["right"]
        out[i] = nd["p1"]
    return out


def tree_predict(node, X):
    return (tree_score(node, X) > 0.5).astype(int)

## Step 4: k nearest neighbours
To classify a new point, look at the k closest training points (straight-line distance)
and take a majority vote. Small k follows every wiggle; large k smooths the boundary.

In [ ]:
def knn_score(X_train, y_train, X_query, k):
    """Fraction of the k nearest training points that are class 1."""
    X_train, X_query = np.asarray(X_train, float), np.asarray(X_query, float)
    d2 = ((X_query[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=2)
    idx = np.argsort(d2, axis=1, kind="stable")[:, :k]
    return np.asarray(y_train)[idx].mean(axis=1)


def knn_predict(X_train, y_train, X_query, k):
    return (knn_score(X_train, y_train, X_query, k) > 0.5).astype(int)


def grid_predict(predict_fn, xlim=(-1.8, 2.8), ylim=(-1.4, 1.8), n=60):
    gx, gy = np.meshgrid(np.linspace(*xlim, n), np.linspace(*ylim, n))
    pts = np.stack([gx.ravel(), gy.ravel()], axis=1)
    return gx, gy, predict_fn(pts).reshape(n, n)


def roughness(grid):
    """Number of neighbouring grid cells whose predictions differ: a measure of boundary length."""
    return int(np.sum(grid[:, 1:] != grid[:, :-1]) + np.sum(grid[1:, :] != grid[:-1, :]))

## Step 5: the confusion matrix and four metrics
                      predicted 1        predicted 0
    actually 1        TP (hit)           FN (miss)
    actually 0        FP (false alarm)   TN (correct rejection)

    accuracy  = (TP + TN) / all            precision = TP / (TP + FP)
    recall    = TP / (TP + FN)             F1 = 2 * precision * recall / (precision + recall)

In [ ]:
def confusion(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fp, fn, tn


def metrics(tp, fp, fn, tn):
    """accuracy, precision, recall, F1. A ratio with a zero denominator is reported as 0."""
    n = tp + fp + fn + tn
    acc = (tp + tn) / n if n else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

## Step 6: the base-rate trap
Bayes' rule for a test with sensitivity (true-positive rate) and specificity
(true-negative rate) on a population with prevalence q:

    PPV = q * sens / ( q * sens + (1 - q) * (1 - spec) )

PPV (positive predictive value) is the chance that a positive result is a true case.

In [ ]:
def ppv(prevalence, sensitivity, specificity):
    tp = prevalence * sensitivity
    fp = (1 - prevalence) * (1 - specificity)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

## Step 7: choosing a threshold when the two mistakes cost different amounts
Score each case between 0 and 1 and call it positive when score >= threshold.
Total cost = c_fn * (misses) + c_fp * (false alarms). Try every threshold, keep the cheapest.
Hand rule: if the score were the true probability p, predicting positive is cheaper when
(1 - p) * c_fp < p * c_fn, i.e. when p > c_fp / (c_fp + c_fn).

In [ ]:
def total_cost(y_true, scores, threshold, c_fn, c_fp):
    pred = (np.asarray(scores) >= threshold).astype(int)
    tp, fp, fn, tn = confusion(y_true, pred)
    return c_fn * fn + c_fp * fp


def bayes_threshold(c_fn, c_fp):
    """If scores are true probabilities, call positive when the score exceeds this value."""
    return c_fp / (c_fp + c_fn)


def best_threshold(y_true, scores, c_fn, c_fp):
    """Try each distinct score (and one above the top) as a threshold; return (threshold, cost).
    Ties go to the larger threshold."""
    cands = np.unique(np.r_[np.asarray(scores, float), np.max(scores) + 1.0])
    costs = [total_cost(y_true, scores, t, c_fn, c_fp) for t in cands]
    i = len(cands) - 1 - int(np.argmin(costs[::-1]))
    return float(cands[i]), float(costs[i])

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    Xtr, ytr = make_moons(100, 0.3, seed=0)
    Xte, yte = make_moons(100, 0.3, seed=1)
    print("gini 50/50:", gini([0, 0, 1, 1]), " gini 3:1:", gini([0, 0, 0, 1]))
    print("best split of 1..4:", best_split(np.array([[1.0], [2.0], [3.0], [4.0]]), np.array([0, 0, 1, 1])))

    T1, T3 = build_tree(Xtr, ytr, 1), build_tree(Xtr, ytr, 3)
    models = {
        "stump (depth 1)": lambda P: tree_predict(T1, P),
        "tree depth 3": lambda P: tree_predict(T3, P),
        "kNN k = 1": lambda P: knn_predict(Xtr, ytr, P, 1),
        "kNN k = 15": lambda P: knn_predict(Xtr, ytr, P, 15),
    }
    fig, axs = plt.subplots(2, 2, figsize=(8, 7), sharex=True, sharey=True)
    for ax, (name, fn) in zip(axs.ravel(), models.items()):
        gx, gy, g = grid_predict(fn)
        ax.contourf(gx, gy, g, levels=[-0.5, 0.5, 1.5], colors=["#cfe8cf", "#f4cccc"])
        ax.scatter(*Xtr[ytr == 0].T, c="tab:green", s=10)
        ax.scatter(*Xtr[ytr == 1].T, c="tab:red", s=10)
        acc = metrics(*confusion(yte, fn(Xte)))["accuracy"]
        ax.set_title(f"{name}: test accuracy {acc:.3f}\nboundary roughness {roughness(g)}", fontsize=9)
        print(f"{name:16s} test accuracy {acc:.3f}  roughness {roughness(g)}")
    for ax in axs[1]:
        ax.set_xlabel("feature 1 (units)")
    for ax in axs[:, 0]:
        ax.set_ylabel("feature 2 (units)")
    fig.tight_layout(); fig.savefig(OUT / "boundaries.png", dpi=120)

    tp, fp, fn_, tn = 40, 10, 20, 130
    m = metrics(tp, fp, fn_, tn)
    print("hand example", {k: round(v, 4) for k, v in m.items()})
    fig2, ax2 = plt.subplots(figsize=(4.5, 4))
    cm = np.array([[tp, fn_], [fp, tn]])
    ax2.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax2.text(j, i, f"{['TP', 'FN', 'FP', 'TN'][2 * i + j]}\n{v}", ha="center", va="center", fontsize=12)
    ax2.set_xticks([0, 1], ["predicted 1", "predicted 0"]); ax2.set_yticks([0, 1], ["actually 1", "actually 0"])
    ax2.set_title("Confusion matrix (hand example, 200 cases)")
    fig2.tight_layout(); fig2.savefig(OUT / "confusion_matrix.png", dpi=120)

    print("PPV 1%, 99%, 95%:", round(ppv(0.01, 0.99, 0.95), 4))
    q = np.linspace(0.001, 0.5, 200)
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    for sp in (0.90, 0.95, 0.99):
        ax3.plot(100 * q, [100 * ppv(v, 0.99, sp) for v in q], label=f"specificity {sp:.0%}")
    ax3.axvline(1, color="gray", ls=":")
    ax3.set_xlabel("prevalence (% of people who have the condition)")
    ax3.set_ylabel("positive predictive value (%)")
    ax3.set_title("A positive result means little when the condition is rare")
    ax3.legend(); ax3.grid(alpha=0.3)
    fig3.tight_layout(); fig3.savefig(OUT / "base_rate.png", dpi=120)

    # threshold design: choose on a validation set (seed 2), then report the cost on the test set (seed 1)
    Xva, yva = make_moons(100, 0.3, seed=2)
    sv, st = knn_score(Xtr, ytr, Xva, 15), knn_score(Xtr, ytr, Xte, 15)
    for c_fn in (1, 10):
        t, c = best_threshold(yva, sv, c_fn, 1)
        print(f"cost of a miss {c_fn}: threshold {t:.3f} chosen on validation; test cost {total_cost(yte, st, t, c_fn, 1):.0f}, test cost at 0.5: {total_cost(yte, st, 0.5, c_fn, 1):.0f}")

## Coding tasks
1. **Easy.** Use `gini` to compute the impurity of a group with 1 positive in 5 cases, then check it by hand: 1 - 0.8^2 - 0.2^2.
2. **Stretch.** Use `ppv` to find the specificity needed for a PPV of at least 90 percent at 1 percent prevalence with sensitivity 99 percent. Try values and explain the pattern.
3. **Explain.** Explain in your own words why the stump has test accuracy 0.745 on curved data while the depth-3 tree reaches 0.865, and what would go wrong if you kept adding depth (Session 3 will measure it).

## Assessment
- **A. Conceptual.** A friend says: 'The test is 95 percent accurate, so a positive result means a 95 percent chance of having the condition.' What is wrong with this?
- **B. Numerical.** A classifier produces TP = 40, FP = 10, FN = 20 and TN = 130 on 200 cases. Compute accuracy, precision, recall and F1.
- **C. Graph and simulation interpretation.** On the PPV graph at prevalence 1 percent, what PPV do specificities of 90, 95 and 99 percent give (sensitivity 99 percent)? At what prevalence does a 95 percent specificity reach a PPV above 80 percent?
- **D. Design and optimisation challenge.** Design: a miss costs 10 units and a false alarm costs 1 unit. You have a classifier that gives scores between 0 and 1. Why should the threshold be below 0.5, how low would the theory put it if the scores were true probabilities, and how should you choose it in practice?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # Gini on hand-checkable cases
    checks.append(("Gini of a 50/50 node is 0.5", abs(L.gini([0, 0, 1, 1]) - 0.5) < 1e-12))
    checks.append(("Gini of a pure node is 0", L.gini([1, 1, 1]) == 0.0 and L.gini([0, 0]) == 0.0))
    checks.append(("Gini of a 3:1 node is 0.375", abs(L.gini([0, 0, 0, 1]) - 0.375) < 1e-12))
    checks.append(("Gini of an empty node is 0 (edge case)", L.gini([]) == 0.0))

    # best split on 1..4 with labels 0,0,1,1: threshold 2.5, weighted Gini 0
    j, t, g = L.best_split(np.array([[1.0], [2.0], [3.0], [4.0]]), np.array([0, 0, 1, 1]))
    checks.append(("best split of [1,2,3,4] with labels [0,0,1,1] is at 2.5 with Gini 0", j == 0 and abs(t - 2.5) < 1e-12 and abs(g) < 1e-12))
    j2, t2, g2 = L.best_split(np.array([[1.0], [1.0], [1.0], [1.0]]), np.array([0, 0, 1, 1]))
    checks.append(("no split exists when all values are equal", j2 is None and abs(g2 - 0.5) < 1e-12))
    # weighted Gini by hand: split [0,0,1 | 1] gives (3 * 4/9 + 1 * 0) / 4 = 1/3
    Xh, yh = np.array([[1.0], [2.0], [3.0], [4.0]]), np.array([0, 0, 1, 1])
    m = Xh[:, 0] <= 3.5
    wg = (m.sum() * L.gini(yh[m]) + (~m).sum() * L.gini(yh[~m])) / 4
    checks.append(("weighted Gini of the split [0,0,1 | 1] is 1/3 (hand calculation)", abs(wg - 1 / 3) < 1e-12))

    # trees and kNN on seeded moons
    Xtr, ytr = L.make_moons(100, 0.3, seed=0)
    Xte, yte = L.make_moons(100, 0.3, seed=1)
    T1, T3 = L.build_tree(Xtr, ytr, 1), L.build_tree(Xtr, ytr, 3)
    acc = lambda pred: L.metrics(*L.confusion(yte, pred))["accuracy"]
    a_stump, a_tree = acc(L.tree_predict(T1, Xte)), acc(L.tree_predict(T3, Xte))
    a_k1, a_k15 = acc(L.knn_predict(Xtr, ytr, Xte, 1)), acc(L.knn_predict(Xtr, ytr, Xte, 15))
    checks.append((f"test accuracy: stump 0.745, depth-3 tree 0.865 (got {a_stump:.3f}, {a_tree:.3f})", abs(a_stump - 0.745) < 1e-9 and abs(a_tree - 0.865) < 1e-9))
    checks.append((f"test accuracy: kNN k=1 0.830, k=15 0.910 (got {a_k1:.3f}, {a_k15:.3f})", abs(a_k1 - 0.83) < 1e-9 and abs(a_k15 - 0.91) < 1e-9))
    checks.append(("a deeper tree beats the stump on this curved data", a_tree > a_stump))
    checks.append(("a depth-0 tree is a single leaf holding the class fraction", L.build_tree(Xtr, ytr, 0)["leaf"] and abs(L.build_tree(Xtr, ytr, 0)["p1"] - 0.5) < 1e-12))
    checks.append(("a stump has exactly two leaves", (not T1["leaf"]) and T1["left"]["leaf"] and T1["right"]["leaf"]))
    # kNN by hand: query 0.1 with training 0,0.2 (class 0) and 1 (class 1), k=1 -> class 0; k=3 -> 1/3 -> class 0
    Xk, yk = np.array([[0.0, 0], [0.2, 0], [1.0, 0]]), np.array([0, 0, 1])
    checks.append(("kNN by hand: k=1 and k=3 both call the point 0.1 class 0", int(L.knn_predict(Xk, yk, np.array([[0.1, 0]]), 1)[0]) == 0 and abs(L.knn_score(Xk, yk, np.array([[0.1, 0]]), 3)[0] - 1 / 3) < 1e-12))
    r1 = L.roughness(L.grid_predict(lambda P: L.knn_predict(Xtr, ytr, P, 1))[2])
    r15 = L.roughness(L.grid_predict(lambda P: L.knn_predict(Xtr, ytr, P, 15))[2])
    checks.append((f"the boundary is smoother for k=15 (roughness {r15}) than k=1 ({r1}): 115 versus 257", r15 < r1 and r1 == 257 and r15 == 115))

    # confusion matrix and metrics: TP 40, FP 10, FN 20, TN 130
    y_true = np.r_[np.ones(60, int), np.zeros(140, int)]
    y_pred = np.r_[np.ones(40, int), np.zeros(20, int), np.ones(10, int), np.zeros(130, int)]
    checks.append(("confusion counts are TP 40, FP 10, FN 20, TN 130", L.confusion(y_true, y_pred) == (40, 10, 20, 130)))
    mt = L.metrics(40, 10, 20, 130)
    checks.append((f"worked example: accuracy 0.85, precision 0.8, recall 0.667, F1 0.727 (got {mt['accuracy']:.3f}, {mt['precision']:.3f}, {mt['recall']:.3f}, {mt['f1']:.3f})",
                   abs(mt["accuracy"] - 0.85) < 1e-12 and abs(mt["precision"] - 0.8) < 1e-12 and abs(mt["recall"] - 2 / 3) < 1e-12 and abs(mt["f1"] - 80 / 110) < 1e-12))
    z = L.metrics(0, 0, 5, 5)
    checks.append(("no predicted positives: precision and F1 reported as 0, no division error", z["precision"] == 0.0 and z["f1"] == 0.0 and abs(z["accuracy"] - 0.5) < 1e-12))

    # base-rate trap
    q = L.ppv(0.01, 0.99, 0.95)
    checks.append((f"worked example: prevalence 1%, sensitivity 99%, specificity 95% gives PPV 16.7% (got {100 * q:.2f}%)", abs(q - 99 / 594) < 1e-12 and abs(q - 0.1667) < 1e-4))
    pop = 10000
    tp, fn_ = pop * 0.01 * 0.99, pop * 0.01 * 0.01
    fp, tn = pop * 0.99 * 0.05, pop * 0.99 * 0.95
    checks.append(("per 10000 people: 99 true positives, 1 miss, 495 false alarms, 9405 true negatives", (round(tp), round(fn_), round(fp), round(tn)) == (99, 1, 495, 9405)))
    checks.append(("overall accuracy of that test is 95.04% yet most positives are false", abs((tp + tn) / pop - 0.9504) < 1e-9 and fp > tp))
    checks.append(("PPV at 1% prevalence: 9.1% for specificity 90%, 50% for 99%; 83% at 20% prevalence with specificity 95%",
                   abs(L.ppv(0.01, 0.99, 0.90) - 0.0909) < 1e-4 and abs(L.ppv(0.01, 0.99, 0.99) - 0.5) < 1e-12 and abs(L.ppv(0.2, 0.99, 0.95) - 0.832) < 1e-3))
    checks.append(("PPV rises with prevalence", L.ppv(0.2, 0.99, 0.95) > L.ppv(0.01, 0.99, 0.95)))
    checks.append(("PPV is 1 for a perfectly specific test and 0 when there are no true cases", abs(L.ppv(0.01, 0.99, 1.0) - 1.0) < 1e-12 and L.ppv(0.0, 0.99, 0.95) == 0.0))

    # threshold design
    checks.append(("hand rule: cost 10 per miss and 1 per false alarm gives threshold 1/11 = 0.0909", abs(L.bayes_threshold(10, 1) - 1 / 11) < 1e-12))
    rng = np.random.default_rng(0)
    p = rng.uniform(0, 1, 20000)
    ysim = (rng.random(20000) < p).astype(int)
    tb, _ = L.best_threshold(ysim, p, 10, 1)
    checks.append((f"searched threshold on true probabilities is close to 1/11 (got {tb:.3f})", abs(tb - 1 / 11) < 0.03))
    Xva, yva = L.make_moons(100, 0.3, seed=2)
    sv, st = L.knn_score(Xtr, ytr, Xva, 15), L.knn_score(Xtr, ytr, Xte, 15)
    thr, _ = L.best_threshold(yva, sv, 10, 1)
    c_tuned, c_half = L.total_cost(yte, st, thr, 10, 1), L.total_cost(yte, st, 0.5, 10, 1)
    checks.append((f"design: a miss costs 10, a false alarm 1: threshold 4/15 = 0.267 gives test cost 55 versus 90 at 0.5 (got {thr:.3f}, {c_tuned:.0f}, {c_half:.0f})",
                   abs(thr - 4 / 15) < 1e-9 and c_tuned == 55 and c_half == 90))
    checks.append(("equal costs: the cost is the same as 'number of mistakes'", L.total_cost(yte, st, 0.5, 1, 1) == len(yte) - sum(1 for a, b in zip(yte, st >= 0.5) if a == int(b))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
